Before getting started with this notebook, make sure you have followed the setup
instructions in the [README.md](README.md#setup) and downloaded at least one
pre-processed file, e.g. `N2a_allPlates_postQC_featureSelected.h5ad`.

Note: to make this notebook easier to run on devices without large RAM, it uses
AnnData's `backed` mode to load in data only when required. This does make
processing slower. If you wish to use the faster but more memory intensive
processing option, change `save_memory` to `False` in the first code cell.

For a simpler sample notebook on how to do hit calling with scmorph, see [here](https://scmorph.readthedocs.io/en/latest/tutorials/basics/hit_calling.html).

In [ ]:
# check that an input file exists
import os
adata_file = "profiles/N2a_allPlates_postQC_featureSelected.h5ad"
save_memory = True
assert os.path.exists(adata_file), f"anndata file {adata_file} not found."

In [ ]:
# load dependencies
import scmorph as sm
import scanpy as sc
import numpy as np
import pandas as pd
from plotnine import *
import matplotlib.pyplot as plt
from natsort import natsorted

In [ ]:
# Set plotting parameters
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9

In [ ]:
adata = sm.read(adata_file, backed="r" if save_memory else None)
adatas_per_plate = {}
for plate in adata.obs["PlateID"].unique():
    adata_ss = adata[adata.obs["PlateID"] == plate]
    adatas_per_plate[plate] = adata_ss

# Perform hit calling

In [ ]:
# Define control wells present across all plates
ctrl_wells = ['E23', 'E24', 'F23', 'F24', 'I23', 'I24', 'J23', 'J24']

In [ ]:
# Test treatments' significance per plate
ref_ks_d = {}
treat_ks_d = {}
for plate in adatas_per_plate:
    ref_ks_d[plate] = {}
    treat_ks_d[plate] = {}
    adata_ss = adatas_per_plate[plate]
    ref_ks_d[plate], treat_ks_d[plate] = sm.tl.get_ks(adata_ss,
                                                        treatment_key="treatment",
                                                        control_wells=ctrl_wells,
                                                        progress=False,
                                                        n_pcs=10,
                                                        scale_by_var=True)

In [ ]:
# Add metadata to resulting dataframes
for plate in ref_ks_d:
    plate_layout = adatas_per_plate[plate].obs["PlateLayout"].values[0]
    rep = adatas_per_plate[plate].obs["Replicate"].values[0]
    ref_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    ref_ks_d[plate].insert(0, "Replicate", rep)
    treat_ks_d[plate].insert(0, "PlateLayout", plate_layout)
    treat_ks_d[plate].insert(0, "Replicate", rep)

In [ ]:
# Collect dataframes across all plates
ref_ks = pd.concat([ref_ks_d[rep] for rep in ref_ks_d]).dropna()
treat_ks = pd.concat([treat_ks_d[rep] for rep in treat_ks_d]).dropna()

In [ ]:

sig_mapping = {0: "NS", 1: "10%", 2: "5%"}

treat_ks["significance"] = (treat_ks["is_significant_0.05"].astype(int) + treat_ks["is_significant_0.1"]).map(sig_mapping)
treat_ks["significance"] = treat_ks["significance"].astype("category").cat.set_categories(["NS", "10%", "5%"], ordered=True)
treat_ks["plate"] = treat_ks["PlateLayout"].str[1:].astype(int).astype("category")

ref_ks["significance"] = (ref_ks["is_significant_0.05"].astype(int) + ref_ks["is_significant_0.1"]).map(sig_mapping)
ref_ks["significance"] = ref_ks["significance"].astype("category").cat.set_categories(["NS", "10%", "5%"], ordered=True)
ref_ks["plate"] = ref_ks["PlateLayout"].str[1:].astype(int).astype("category")

In [ ]:
treat_ks